# Day 1, Part 1: NumPy and pandas on the Titanic data

**Goal:** get comfortable with the data before any model touches it. This notebook covers NumPy basics, pandas analysis, cleaning, and a saved `titanic_clean.csv` that Part 2 uses.

**Before you start**
- Keep `train.csv` and this notebook in the same folder.
- Use **Run All**. After a kernel restart, run the imports cell first, or you'll see `NameError: name 'np' is not defined`.
- Never name your own file `numpy.py` or `pandas.py`, because Python would import your file instead of the library.
- If a library is missing: `pip install numpy pandas matplotlib`


In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
print("NumPy", np.__version__, "| pandas", pd.__version__)

NumPy 2.5.3 | pandas 3.0.5


## 1. First look

Each row is one passenger. `Survived` is the target, the thing a model will later predict. Everything else is a candidate feature.

| Column | Meaning |
|---|---|
| `Pclass` | Ticket class: 1 = upper, 3 = lower |
| `SibSp` | Siblings and spouses aboard |
| `Parch` | Parents and children aboard |
| `Fare` | Ticket price (pounds) |
| `Embarked` | Port: C = Cherbourg, Q = Queenstown, S = Southampton |


In [2]:
df = pd.read_csv("train.csv")
print(df.shape)
df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [4]:
df.describe().round(2)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,891.00,714.00,891.00,891.00,891.00
mean,446.00,0.38,2.31,29.70,0.52,0.38,32.20
std,257.35,0.49,0.84,14.53,1.10,0.81,49.69
min,1.00,0.00,1.00,0.42,0.00,0.00,0.00
25%,223.50,0.00,2.00,20.12,0.00,0.00,7.91
50%,446.00,0.00,3.00,28.00,0.00,0.00,14.45
75%,668.50,1.00,3.00,38.00,1.00,0.00,31.00
max,891.00,1.00,3.00,80.00,8.00,6.00,512.33


## 2. NumPy: fast maths on arrays

A pandas column is backed by an array. NumPy works on the whole array at once, so you rarely need a `for` loop.

In [5]:
survived = df["Survived"].to_numpy()
fare = df["Fare"].to_numpy()

print("Passengers:      ", survived.size)
print("Survival rate:   ", round(survived.mean(), 4))
print("Median fare:     ", np.median(fare))
print("Fare std dev:    ", round(fare.std(), 2))

Passengers:       891
Survival rate:    0.3838
Median fare:      14.4542
Fare std dev:     49.67


### Boolean masks

A comparison gives an array of `True`/`False`. Using it as an index keeps only the `True` rows. The mean of 0s and 1s is a rate, so `survived[mask].mean()` is a survival rate. You'll use this pattern constantly.

In [6]:
is_female = df["Sex"].to_numpy() == "female"

print("Women survived:", round(survived[is_female].mean(), 4))
print("Men survived:  ", round(survived[~is_female].mean(), 4))

Women survived: 0.742
Men survived:   0.1889


### Linear algebra: correlation is a dot product

This is why NumPy matters for ML. Convert two columns to z-scores (mean 0, standard deviation 1), and their Pearson correlation is just their dot product divided by the row count.

$$r = \frac{1}{n}\, z_x \cdot z_y \qquad\text{where}\qquad z = \frac{x - \bar{x}}{\sigma}$$

The same `w · x` operation is the heart of linear and logistic regression on Day 2.

In [7]:
def z_score(x):
    return (x - x.mean()) / x.std()

r_by_hand = z_score(fare) @ z_score(survived) / fare.size
r_numpy = np.corrcoef(fare, survived)[0, 1]

print("By hand:      ", round(r_by_hand, 4))
print("np.corrcoef:  ", round(r_numpy, 4))

By hand:       0.2573
np.corrcoef:   0.2573


### Matrices

Stack features into a matrix `X` with one row per passenger and one column per feature. Then `X @ w` scores all passengers in one step. The weights below are invented purely to show the mechanics. Training finds real weights from Day 2 onward.

In [8]:
X = df[["Pclass", "Fare"]].to_numpy()
w = np.array([-0.5, 0.01])

scores = X @ w
print("X shape:     ", X.shape)
print("w shape:     ", w.shape)
print("scores shape:", scores.shape)
print("First 3 scores:", scores[:3].round(3))

X shape:      (891, 2)
w shape:      (2,)
scores shape: (891,)
First 3 scores: [-1.428  0.213 -1.421]


## 3. pandas: answering questions about the data

### What's missing?

In [9]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "percent": (df.isna().mean() * 100).round(1),
})
missing[missing["missing"] > 0].sort_values("missing", ascending=False)

,missing,percent
Cabin,687,77.1
Age,177,19.9
Embarked,2,0.2


- **Cabin** is 77.1% empty, so its values are too sparse to fill. Whether a cabin was recorded may still carry signal.
- **Age** is 19.9% empty, which is worth filling carefully.
- **Embarked** has only 2 gaps, so the most common value is fine.

### Who survived?

`groupby` splits the rows, applies a calculation to each group, and combines the results.

In [10]:
df.groupby("Sex")["Survived"].agg(["mean", "count"]).round(3)

,mean,count
Sex,,
female,0.742,314
male,0.189,577


In [11]:
df.groupby("Pclass")["Survived"].agg(["mean", "count"]).round(3)

,mean,count
Pclass,,
1,0.630,216
2,0.473,184
3,0.242,491


In [12]:
df.pivot_table(index="Sex", columns="Pclass", values="Survived").round(3)

Pclass,1,2,3
Sex,,,
female,0.968,0.921,0.500
male,0.369,0.157,0.135


The pivot table says more than either table alone. First- and second-class women almost all survived. Third-class women were close to a coin flip, and men in every class fared worse than third-class women.

## 4. Cleaning and feature engineering

**Age:** a single overall median would ignore the fact that age differs by class and sex, so we fill with the median of each `Pclass` + `Sex` group.

In [13]:
df.groupby(["Pclass", "Sex"])["Age"].median()

Pclass  Sex   
1       female    35.0
        male      40.0
2       female    28.0
        male      30.0
3       female    21.5
        male      25.0
Name: Age, dtype: float64

In [14]:
clean = df.copy()

# Keep a record of which ages were real. Part 2 plots only these.
clean["AgeMissing"] = clean["Age"].isna().astype(int)

# Fill the gaps
clean["Age"] = clean["Age"].fillna(clean.groupby(["Pclass", "Sex"])["Age"].transform("median"))
clean["Embarked"] = clean["Embarked"].fillna(clean["Embarked"].mode()[0])

# New features
clean["HasCabin"] = clean["Cabin"].notna().astype(int)
clean["FamilySize"] = clean["SibSp"] + clean["Parch"] + 1
clean["IsAlone"] = (clean["FamilySize"] == 1).astype(int)
clean["LogFare"] = np.log1p(clean["Fare"])
clean["SexFemale"] = (clean["Sex"] == "female").astype(int)

# Title sits between the comma and the full stop: "Braund, Mr. Owen Harris"
title = clean["Name"].str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
clean["Title"] = title.where(title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

clean.isna().sum()[lambda s: s > 0]

Cabin    687
dtype: int64

Only `Cabin` still has gaps, and `HasCabin` now covers what we need from it.

**Why these features?**
- `FamilySize` and `IsAlone` combine `SibSp` and `Parch` into something more meaningful.
- `LogFare` squashes the long right tail of `Fare` (skew about 4.8), which helps linear models later.
- `Title` packs age and sex together. `Master` means a young boy.

**A warning for Day 3:** here we calculated the medians from all the rows, which is fine for exploring. In a real model that leaks information, so those values must be learned from the training folds only. That's what `Pipeline` is for.

In [15]:
clean.groupby("Title")["Survived"].agg(["mean", "count"]).round(3).sort_values("mean", ascending=False)

,mean,count
Title,,
Mrs,0.794,126
Miss,0.703,185
Master,0.575,40
Rare,0.348,23
Mr,0.157,517


In [16]:
clean.to_csv("titanic_clean.csv", index=False)
print("Saved titanic_clean.csv:", clean.shape)

Saved titanic_clean.csv: (891, 19)


## 5. Your turn

Try these without scrolling down to the answers. Add a new cell for each.

1. **NumPy:** use a boolean mask to find the survival rate of passengers with a known age under 10. Remember that `NaN < 10` is `False`, so missing ages drop out by themselves.
2. **pandas:** find the survival rate and passenger count for each `Embarked` port in `clean`.
3. **pandas:** which `FamilySize` has the highest survival rate among groups with at least 20 people?

**Answers (no peeking):** (1) 61.3% of 62 children. (2) C: 55.4% of 168, Q: 39.0% of 77, S: 33.9% of 646. (3) Family size 4, at 72.4% of 29.

**Done?** Open `titanic2.ipynb`.
